# Camera Discovery Live Test

Simplified 3-stage pipeline: `TargetResolver → CandidateDiscoveryEngine → ReviewAndValidationPipeline`.

LLMs are used as advisory evidence interpreters/rankers for target intent, geocoder candidate ranking, and candidate semantic review. Deterministic code/tools remain responsible for geometry verification, stream validation, trusted-output authorization, and final artifact writing.

This notebook clones the configured GitHub repository branch by default, installs it in editable mode with the optional Playwright extra, installs a headless Chromium browser for dynamic-page network capture, runs a live test, and then displays trusted or untrusted camera outputs. It does not patch source files from notebook cells.

Coordinate enrichment uses source coordinates first, then candidate metadata/title geocoding, then the optional LLM location-inference fallback. The LLM fallback only infers place-name query variants from stream URLs and metadata; Nominatim supplies coordinates, and verified target bounding boxes still decide whether inferred coordinates are accepted.

This notebook supports single-location and multi-location queries, for example:

```text
Get me all traffic cameras from California
Get me all cameras from Greenville, Texas
Get me all cameras from London, England and New York, New York
```

The query is intentionally user-controlled. Phrases such as `traffic cameras`, `weather cameras`, or `public live cameras` should be interpreted as camera-type intent, while place names such as `California`, `Greenville, Texas`, or `London, England` are target geography.


Dynamic camera pages are supported when a source row uses `type: dynamic`; the notebook installs Playwright so those rows can capture `.m3u8`, JSON feed, MapServer, FeatureServer, ArcGIS, and camera API network requests during real browser rendering.


The CLI run uses the application progress reporting from `camera_discovery.cli run`. In notebooks, it launches the CLI inside a pseudo-terminal so Rich progress bars update in place instead of printing repeated frames or noisy milestone lines. The combined CLI log is still saved without patching repository source code.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json
import shutil
import pty
import select

# Colab/repo bootstrap settings. Override with env vars if needed.
REPO_URL = os.environ.get("CAMERA_DISCOVERY_REPO_URL", "https://github.com/dshipley71/camera-discovery.git")
REPO_BRANCH = os.environ.get("CAMERA_DISCOVERY_REPO_BRANCH", "main")
REPO_DIR = Path(os.environ.get("CAMERA_DISCOVERY_REPO_DIR", "/content/camera-discovery"))

print("Notebook bootstrap")
print("repo url:", REPO_URL)
print("branch:", REPO_BRANCH)
print("repo dir:", REPO_DIR)

# No source files are patched by this notebook. To test changes, push/update the GitHub branch
# selected above and rerun the clone/install cells.


In [ ]:
%cd /content

if REPO_DIR.exists():
    print(f"Removing existing repo directory: {REPO_DIR}")
    shutil.rmtree(REPO_DIR)

clone_cmd = ["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)]
print("$", " ".join(clone_cmd))
subprocess.run(clone_cmd, check=True)

os.chdir(REPO_DIR)
print("cwd:", Path.cwd())

src_path = REPO_DIR / "src"
assert (src_path / "camera_discovery").exists(), f"Missing package at {src_path / 'camera_discovery'}"

# Make imports work immediately, even before editable install finishes.
os.environ["PYTHONPATH"] = str(src_path)
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Install the package with the optional Playwright extra. Playwright is not a
# default dependency of the package, but this notebook enables it so dynamic
# source rows can perform real browser/network capture for .m3u8 and feed URLs.
install_cmd = [sys.executable, "-m", "pip", "install", "-e", ".[playwright]", "--no-build-isolation"]
print("$", " ".join(install_cmd))
subprocess.run(install_cmd, check=True)

# Install Chromium for Playwright. In Colab this is required before dynamic-page
# capture can launch a headless browser. Set CAMERA_DISCOVERY_INSTALL_PLAYWRIGHT_BROWSER=false
# only if the browser is already installed in your runtime.
INSTALL_PLAYWRIGHT_BROWSER = os.environ.get("CAMERA_DISCOVERY_INSTALL_PLAYWRIGHT_BROWSER", "true").strip().lower() in {"1", "true", "yes", "on"}
if INSTALL_PLAYWRIGHT_BROWSER:
    browser_cmd = [sys.executable, "-m", "playwright", "install", "chromium"]
    print("$", " ".join(browser_cmd))
    subprocess.run(browser_cmd, check=True)
else:
    print("Skipping Playwright browser install because CAMERA_DISCOVERY_INSTALL_PLAYWRIGHT_BROWSER=false")

# Verify Playwright imports before the live run. Dynamic capture remains optional:
# if no SOURCES.md row uses type: dynamic, the browser is not launched.
try:
    from playwright.sync_api import sync_playwright  # type: ignore
    print("Playwright import OK; dynamic source rows can use browser network capture.")
except Exception as exc:
    raise RuntimeError(f"Playwright import failed after installation: {exc!r}")


In [ ]:
import camera_discovery
print("camera_discovery import OK:", camera_discovery.__file__)

# Load provider secrets from Colab userdata when available.
# Configure these in Colab as needed:
#   OLLAMA_API_KEY
#   OPENAI_API_KEY
#   OPENAI_BASE_URL
#   AWS_ACCESS_KEY_ID
#   AWS_SECRET_ACCESS_KEY
#   AWS_SESSION_TOKEN
#   AWS_DEFAULT_REGION
try:
    from google.colab import userdata  # type: ignore
except Exception as exc:
    userdata = None
    print("Colab userdata not available:", repr(exc))

if userdata is not None:
    for key in [
        "OLLAMA_API_KEY",
        "OPENAI_API_KEY",
        "OPENAI_BASE_URL",
        "AWS_ACCESS_KEY_ID",
        "AWS_SECRET_ACCESS_KEY",
        "AWS_SESSION_TOKEN",
        "AWS_DEFAULT_REGION",
    ]:
        if os.environ.get(key):
            continue
        try:
            value = userdata.get(key)
        except Exception:
            value = None
        if value:
            os.environ[key] = value
            print(f"Loaded {key} from Colab userdata")


# Sanity-check the installed CLI module without printing help/usage output.
import camera_discovery.cli as camera_cli
print("camera_discovery.cli import OK:", camera_cli.__file__)


| Profile    | Purpose                    | Behavior                                                                                                                                                                             |
| ---------- | -------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| `fast`     | Quick review/discovery run | Validation is minimized/disabled; trusted `camera.geojson` should not be produced unless trust requirements are met; useful for `untrusted_camera_candidates.geojson` review output. |
| `balanced` | Middle-ground run          | More validation than Fast, but avoids the most expensive checks. Good default for routine testing.                                                                                   |
| `full`     | Most thorough run          | Runs the deepest validation path available, intended for trusted output when geometry and stream validation pass.                                                                    |


In [ ]:
RUN_PROFILE = os.environ.get("CAMERA_DISCOVERY_PROFILE", "fast").strip().lower()
if RUN_PROFILE not in {"fast", "balanced", "full"}:
    raise ValueError(f"Invalid CAMERA_DISCOVERY_PROFILE={RUN_PROFILE!r}; expected fast, balanced, or full")

# User-controlled query. Edit this directly or set CAMERA_DISCOVERY_QUERY in the environment.
# Camera-type terms such as "traffic cameras" are camera intent, not target geography.
USER_QUERY = os.environ.get("CAMERA_DISCOVERY_QUERY", "Get me all traffic cameras from California")

# Other valid examples:
# USER_QUERY = "Get me all cameras from Greenville, Texas"
# USER_QUERY = "Get me all cameras from London, England and New York, New York"

OUTPUT_DIR = Path(os.environ.get("CAMERA_DISCOVERY_OUTPUT_DIR", "runs/notebook-live-test"))
CLEAN_OUTPUT_DIR = os.environ.get("CAMERA_DISCOVERY_CLEAN_OUTPUT_DIR", "true").strip().lower() in {"1", "true", "yes", "on"}

# LLM provider/model settings.
# This application requires a real LLM provider. Each application section can use
# a different model. Edit the variables below, or set the matching environment
# variables before running this cell.
LLM_PROVIDER = os.environ.get("CAMERA_DISCOVERY_LLM_PROVIDER", "ollama-cloud").strip()

# Defaults are intentionally independent so users can test different models per stage.
DEFAULT_MAIN_MODEL = "gemma4:31b-cloud"
DEFAULT_TARGET_INTENT_MODEL = "qwen3.5:4b"
DEFAULT_TARGET_INTENT_FALLBACK_MODEL = "qwen3.5:4b"
DEFAULT_GEOCODER_REFEREE_MODEL = "gemma4:31b-cloud"
DEFAULT_LOCATION_INFERENCE_MODEL = "gemma4:31b-cloud"
DEFAULT_CANDIDATE_REVIEW_MODEL = "qwen3.5:4b"

# For example, after confirming availability in your provider account, you can use:
# DEFAULT_TARGET_INTENT_MODEL = "qwen3.5:4b"       # fast extraction
# DEFAULT_GEOCODER_REFEREE_MODEL = "gemma4:31b-cloud" # stronger semantic ranking
# DEFAULT_CANDIDATE_REVIEW_MODEL = "qwen3.5:4b"       # faster batch review

MAIN_MODEL = os.environ.get("CAMERA_DISCOVERY_LLM_MODEL", DEFAULT_MAIN_MODEL).strip()
TARGET_INTENT_MODEL = os.environ.get("CAMERA_DISCOVERY_TARGET_INTENT_MODEL", DEFAULT_TARGET_INTENT_MODEL).strip()
TARGET_INTENT_FALLBACK_MODEL = os.environ.get("CAMERA_DISCOVERY_TARGET_INTENT_FALLBACK_MODEL", DEFAULT_TARGET_INTENT_FALLBACK_MODEL).strip()
TARGET_INTENT_TIMEOUT = os.environ.get("CAMERA_DISCOVERY_TARGET_INTENT_TIMEOUT", "30").strip()
TARGET_INTENT_ATTEMPTS = os.environ.get("CAMERA_DISCOVERY_TARGET_INTENT_ATTEMPTS", "1").strip()
GEOCODER_REFEREE_MODEL = os.environ.get("CAMERA_DISCOVERY_GEOCODER_REFEREE_MODEL", DEFAULT_GEOCODER_REFEREE_MODEL).strip()
LOCATION_INFERENCE_MODEL = os.environ.get("CAMERA_DISCOVERY_LOCATION_INFERENCE_MODEL", DEFAULT_LOCATION_INFERENCE_MODEL).strip()
LOCATION_INFERENCE_TIMEOUT = os.environ.get("CAMERA_DISCOVERY_LOCATION_INFERENCE_TIMEOUT", "45").strip()
ENABLE_LLM_LOCATION_INFERENCE = os.environ.get("CAMERA_DISCOVERY_ENABLE_LLM_LOCATION_INFERENCE", "true").strip().lower()
MAX_LLM_LOCATION_INFERENCES = os.environ.get("CAMERA_DISCOVERY_MAX_LLM_LOCATION_INFERENCES", "75").strip()
LOCATION_INFERENCE_MIN_CONFIDENCE = os.environ.get("CAMERA_DISCOVERY_LOCATION_INFERENCE_MIN_CONFIDENCE", "0.70").strip()
CANDIDATE_REVIEW_MODEL = os.environ.get("CAMERA_DISCOVERY_CANDIDATE_REVIEW_MODEL", DEFAULT_CANDIDATE_REVIEW_MODEL).strip()
CANDIDATE_REVIEW_TIMEOUT = os.environ.get("CAMERA_DISCOVERY_CANDIDATE_REVIEW_TIMEOUT", "60").strip()
CANDIDATE_REVIEW_BATCH_SIZE = os.environ.get("CAMERA_DISCOVERY_CANDIDATE_REVIEW_BATCH_SIZE", "8").strip()
MAX_CANDIDATE_REVIEWS = os.environ.get("CAMERA_DISCOVERY_MAX_CANDIDATE_REVIEWS", "50").strip()
MAX_SEARCH_QUERIES = os.environ.get("CAMERA_DISCOVERY_MAX_SEARCH_QUERIES", "10").strip()
MAX_SEARCH_RESULTS_PER_QUERY = os.environ.get("CAMERA_DISCOVERY_MAX_SEARCH_RESULTS_PER_QUERY", "8").strip()
MAX_PAGES = os.environ.get("CAMERA_DISCOVERY_MAX_PAGES", "60").strip()
MAX_HLS_CANDIDATES = os.environ.get("CAMERA_DISCOVERY_MAX_HLS_CANDIDATES", "300").strip()
MAX_IMAGE_SNAPSHOT_CANDIDATES = os.environ.get("CAMERA_DISCOVERY_MAX_IMAGE_SNAPSHOT_CANDIDATES", "200").strip()
MAX_TOTAL_CANDIDATES = os.environ.get("CAMERA_DISCOVERY_MAX_TOTAL_CANDIDATES", "500").strip()
# Deprecated compatibility cap; keep aligned with the total budget by default.
MAX_STREAMS = os.environ.get("CAMERA_DISCOVERY_MAX_STREAMS", MAX_TOTAL_CANDIDATES).strip()
MAX_DIRECTORY_PAGES = os.environ.get("CAMERA_DISCOVERY_MAX_DIRECTORY_PAGES", "10").strip()
MAX_STRUCTURED_ENDPOINTS_PER_PAGE = os.environ.get("CAMERA_DISCOVERY_MAX_STRUCTURED_ENDPOINTS_PER_PAGE", "25").strip()
MAX_CANDIDATE_GEOCODES = os.environ.get("CAMERA_DISCOVERY_MAX_CANDIDATE_GEOCODES", "75").strip()
MAX_STATE_SCALE_CANDIDATE_GEOCODES = os.environ.get("CAMERA_DISCOVERY_MAX_STATE_SCALE_CANDIDATE_GEOCODES", "300").strip()
IMAGE_SNAPSHOT_REFRESH_DELAY_SECONDS = os.environ.get("CAMERA_DISCOVERY_IMAGE_SNAPSHOT_REFRESH_DELAY_SECONDS", "2.0").strip()

# Preserve independent model choices. Do not normalize all stages to one model.
os.environ["CAMERA_DISCOVERY_LLM_PROVIDER"] = LLM_PROVIDER
os.environ["CAMERA_DISCOVERY_LLM_MODEL"] = MAIN_MODEL
os.environ["CAMERA_DISCOVERY_TARGET_INTENT_MODEL"] = TARGET_INTENT_MODEL
os.environ["CAMERA_DISCOVERY_TARGET_INTENT_FALLBACK_MODEL"] = TARGET_INTENT_FALLBACK_MODEL
os.environ["CAMERA_DISCOVERY_TARGET_INTENT_TIMEOUT"] = TARGET_INTENT_TIMEOUT
os.environ["CAMERA_DISCOVERY_TARGET_INTENT_ATTEMPTS"] = TARGET_INTENT_ATTEMPTS
os.environ["CAMERA_DISCOVERY_GEOCODER_REFEREE_MODEL"] = GEOCODER_REFEREE_MODEL
os.environ["CAMERA_DISCOVERY_LOCATION_INFERENCE_MODEL"] = LOCATION_INFERENCE_MODEL
os.environ["CAMERA_DISCOVERY_LOCATION_INFERENCE_TIMEOUT"] = LOCATION_INFERENCE_TIMEOUT
os.environ["CAMERA_DISCOVERY_ENABLE_LLM_LOCATION_INFERENCE"] = ENABLE_LLM_LOCATION_INFERENCE
os.environ["CAMERA_DISCOVERY_MAX_LLM_LOCATION_INFERENCES"] = MAX_LLM_LOCATION_INFERENCES
os.environ["CAMERA_DISCOVERY_LOCATION_INFERENCE_MIN_CONFIDENCE"] = LOCATION_INFERENCE_MIN_CONFIDENCE
os.environ["CAMERA_DISCOVERY_CANDIDATE_REVIEW_MODEL"] = CANDIDATE_REVIEW_MODEL
os.environ["CAMERA_DISCOVERY_CANDIDATE_REVIEW_TIMEOUT"] = CANDIDATE_REVIEW_TIMEOUT
os.environ["CAMERA_DISCOVERY_CANDIDATE_REVIEW_BATCH_SIZE"] = CANDIDATE_REVIEW_BATCH_SIZE
os.environ["CAMERA_DISCOVERY_MAX_CANDIDATE_REVIEWS"] = MAX_CANDIDATE_REVIEWS
os.environ["CAMERA_DISCOVERY_MAX_SEARCH_QUERIES"] = MAX_SEARCH_QUERIES
os.environ["CAMERA_DISCOVERY_MAX_SEARCH_RESULTS_PER_QUERY"] = MAX_SEARCH_RESULTS_PER_QUERY
os.environ["CAMERA_DISCOVERY_MAX_PAGES"] = MAX_PAGES
os.environ["CAMERA_DISCOVERY_MAX_HLS_CANDIDATES"] = MAX_HLS_CANDIDATES
os.environ["CAMERA_DISCOVERY_MAX_IMAGE_SNAPSHOT_CANDIDATES"] = MAX_IMAGE_SNAPSHOT_CANDIDATES
os.environ["CAMERA_DISCOVERY_MAX_TOTAL_CANDIDATES"] = MAX_TOTAL_CANDIDATES
os.environ["CAMERA_DISCOVERY_MAX_STREAMS"] = MAX_STREAMS
os.environ["CAMERA_DISCOVERY_MAX_DIRECTORY_PAGES"] = MAX_DIRECTORY_PAGES
os.environ["CAMERA_DISCOVERY_MAX_STRUCTURED_ENDPOINTS_PER_PAGE"] = MAX_STRUCTURED_ENDPOINTS_PER_PAGE
os.environ["CAMERA_DISCOVERY_MAX_CANDIDATE_GEOCODES"] = MAX_CANDIDATE_GEOCODES
os.environ["CAMERA_DISCOVERY_MAX_STATE_SCALE_CANDIDATE_GEOCODES"] = MAX_STATE_SCALE_CANDIDATE_GEOCODES
os.environ["CAMERA_DISCOVERY_IMAGE_SNAPSHOT_REFRESH_DELAY_SECONDS"] = IMAGE_SNAPSHOT_REFRESH_DELAY_SECONDS

DISCOVERY_MODE = os.environ.get("CAMERA_DISCOVERY_DISCOVERY_MODE", "both").strip().lower()
if DISCOVERY_MODE not in {"blind", "directory", "both", "direct"}:
    raise ValueError(f"Invalid CAMERA_DISCOVERY_DISCOVERY_MODE={DISCOVERY_MODE!r}")

SOURCES_FILE = Path(os.environ.get("CAMERA_DISCOVERY_SOURCES_FILE", "SOURCES.md"))
SEED_URLS = [url.strip() for url in os.environ.get("CAMERA_DISCOVERY_SEED_URLS", "").split(",") if url.strip()]

required_secret_hint = {
    "ollama": "OLLAMA_API_KEY is required only when using Ollama Cloud; local Ollama may not need it.",
    "ollama-cloud": "OLLAMA_API_KEY is required.",
    "ollama_cloud": "OLLAMA_API_KEY is required.",
    "openai-compatible": "OPENAI_API_KEY and OPENAI_BASE_URL are usually required.",
    "openai_compatible": "OPENAI_API_KEY and OPENAI_BASE_URL are usually required.",
    "bedrock": "AWS credentials and AWS_DEFAULT_REGION are required.",
}.get(LLM_PROVIDER.lower(), "provider-specific credentials are required")

print("profile:", RUN_PROFILE)
print("query:", USER_QUERY)
print("output:", OUTPUT_DIR)
print("clean output dir before run:", CLEAN_OUTPUT_DIR)
print("provider:", LLM_PROVIDER)
print("main model:", MAIN_MODEL)
print("target intent model:", TARGET_INTENT_MODEL)
print("target intent fallback model:", TARGET_INTENT_FALLBACK_MODEL)
print("target intent timeout:", TARGET_INTENT_TIMEOUT)
print("target intent attempts:", TARGET_INTENT_ATTEMPTS)
print("geocoder referee model:", GEOCODER_REFEREE_MODEL)
print("location inference model:", LOCATION_INFERENCE_MODEL)
print("location inference timeout:", LOCATION_INFERENCE_TIMEOUT)
print("enable LLM location inference:", ENABLE_LLM_LOCATION_INFERENCE)
print("max LLM location inferences:", MAX_LLM_LOCATION_INFERENCES)
print("location inference min confidence:", LOCATION_INFERENCE_MIN_CONFIDENCE)
print("candidate review model:", CANDIDATE_REVIEW_MODEL)
print("candidate review timeout:", CANDIDATE_REVIEW_TIMEOUT)
print("candidate review batch size:", CANDIDATE_REVIEW_BATCH_SIZE)
print("max candidate reviews:", MAX_CANDIDATE_REVIEWS)
print("max search queries:", MAX_SEARCH_QUERIES)
print("max search results per query:", MAX_SEARCH_RESULTS_PER_QUERY)
print("max pages:", MAX_PAGES)
print("max hls candidates:", MAX_HLS_CANDIDATES)
print("max image snapshot candidates:", MAX_IMAGE_SNAPSHOT_CANDIDATES)
print("max total candidates:", MAX_TOTAL_CANDIDATES)
print("max streams deprecated compatibility cap:", MAX_STREAMS)
print("max directory pages:", MAX_DIRECTORY_PAGES)
print("max structured endpoints per page:", MAX_STRUCTURED_ENDPOINTS_PER_PAGE)
print("max candidate geocodes:", MAX_CANDIDATE_GEOCODES)
print("max state-scale candidate geocodes:", MAX_STATE_SCALE_CANDIDATE_GEOCODES)
print("image snapshot refresh validation delay seconds:", IMAGE_SNAPSHOT_REFRESH_DELAY_SECONDS)
print("discovery mode:", DISCOVERY_MODE)
print("sources file:", SOURCES_FILE)
print("seed urls:", len(SEED_URLS))
print("credential hint:", required_secret_hint)

if DISCOVERY_MODE == "direct" and not SEED_URLS:
    raise ValueError("direct discovery mode requires CAMERA_DISCOVERY_SEED_URLS or --seed-url values")

if LLM_PROVIDER.lower() in {"ollama-cloud", "ollama_cloud"} and not os.environ.get("OLLAMA_API_KEY"):
    print("WARNING: OLLAMA_API_KEY is not set; Ollama Cloud requests will fail until configured.")


In [ ]:
cmd = [
    sys.executable, "-m", "camera_discovery.cli", "run", USER_QUERY,
    "--profile", RUN_PROFILE,
    "--output-dir", str(OUTPUT_DIR),
    "--discovery-mode", DISCOVERY_MODE,
    "--sources-file", str(SOURCES_FILE),
    "--progress",
    "--progress-style", "rich",
]
for url in SEED_URLS:
    cmd.extend(["--seed-url", url])

# Remove stale run artifacts before each live test unless explicitly disabled.
if CLEAN_OUTPUT_DIR and OUTPUT_DIR.exists():
    resolved_output = OUTPUT_DIR.resolve()
    resolved_repo = REPO_DIR.resolve()
    unsafe_roots = {Path("/").resolve(), Path("/content").resolve(), resolved_repo}
    if resolved_output in unsafe_roots:
        raise RuntimeError(f"Refusing to remove unsafe output directory: {resolved_output}")
    print(f"Removing stale output directory: {resolved_output}")
    shutil.rmtree(resolved_output)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
combined_log = OUTPUT_DIR / "notebook_cli_combined.log"
print("$", " ".join(cmd))
print("Streaming CLI output live; Rich progress bars are rendered through a pseudo-terminal to avoid repeated progress spam.")
print("combined log:", combined_log)

run_env = os.environ.copy()
run_env["PYTHONPATH"] = str(src_path)
run_env.setdefault("TERM", "xterm-256color")
run_env.setdefault("COLUMNS", "140")
# The CLI writes Rich progress bars only when stdout is attached to a terminal.
# A pseudo-terminal lets the bars update in place in notebook output instead of
# being captured as repeated progress lines.
master_fd, slave_fd = pty.openpty()
process = subprocess.Popen(
    cmd,
    stdin=subprocess.DEVNULL,
    stdout=slave_fd,
    stderr=slave_fd,
    env=run_env,
    close_fds=True,
)
os.close(slave_fd)

with combined_log.open("wb") as log_fh:
    while True:
        ready, _, _ = select.select([master_fd], [], [], 0.2)
        if master_fd in ready:
            try:
                chunk = os.read(master_fd, 4096)
            except OSError:
                break
            if not chunk:
                break
            log_fh.write(chunk)
            log_fh.flush()
            try:
                sys.stdout.buffer.write(chunk)
                sys.stdout.flush()
            except AttributeError:
                print(chunk.decode("utf-8", errors="replace"), end="")
        if process.poll() is not None:
            # Drain any remaining output after process exit.
            while True:
                try:
                    chunk = os.read(master_fd, 4096)
                except OSError:
                    chunk = b""
                if not chunk:
                    break
                log_fh.write(chunk)
                log_fh.flush()
                try:
                    sys.stdout.buffer.write(chunk)
                    sys.stdout.flush()
                except AttributeError:
                    print(chunk.decode("utf-8", errors="replace"), end="")
            break
os.close(master_fd)
returncode = process.wait()
print("\nexit:", returncode)
print("combined CLI log:", combined_log)

if returncode != 0:
    raise RuntimeError("camera-discovery run failed; inspect notebook_cli_combined.log and run artifacts")


In [ ]:
from pathlib import Path
import json

for rel in ["logs/source_policy_summary.json", "logs/candidate_discovery_summary.json", "logs/run_summary.json"]:
    path = OUTPUT_DIR / rel
    print("---", rel, "exists=", path.exists())
    if path.exists():
        try:
            print(json.dumps(json.loads(path.read_text(encoding="utf-8")), indent=2)[:4000])
        except Exception as exc:
            print("Could not parse JSON:", repr(exc))
            print(path.read_text(encoding="utf-8")[:1000])


In [ ]:
summary_path = OUTPUT_DIR / "logs" / "run_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.exists() else {}
targets = summary.get("targets", [])
print("targets:", len(targets))
for t in targets:
    print(json.dumps({
        "target_id": t.get("target_id"),
        "target_label": t.get("target_label"),
        "canonical_target": t.get("canonical_target"),
        "geometry_status": t.get("geometry_status"),
        "bbox_verified": t.get("bbox_verified"),
        "trust_policy": t.get("trust_policy"),
    }, indent=2))

candidate_summary = summary.get("candidates", {}) or {}
output_summary = summary.get("outputs", {}) or {}
unique_value = candidate_summary.get("unique_count")
if unique_value is None and isinstance(candidate_summary.get("unique"), list):
    unique_value = len(candidate_summary.get("unique"))
coord_value = candidate_summary.get("coordinate_bearing_count")
if coord_value is None and isinstance(candidate_summary.get("coordinate_bearing"), list):
    coord_value = len(candidate_summary.get("coordinate_bearing"))

print(json.dumps({
    "unique_candidates": unique_value,
    "coordinate_bearing": coord_value,
    "trusted_geojson_features": output_summary.get("trusted_geojson_features_written", 0),
    "untrusted_geojson_features": output_summary.get("untrusted_geojson_features_written", 0),
    "trusted_geojson_created": output_summary.get("trusted_geojson_created"),
    "untrusted_geojson_created": output_summary.get("untrusted_geojson_created"),
}, indent=2))


In [ ]:
from IPython.display import Markdown, display

for rel in [
    'camera.geojson',
    'untrusted_camera_candidates.geojson',
    'map.html',
    'notebook_camera_map.html',
    'review_artifacts.zip',
    'RUN_EXPLANATION.md',
    'logs/run_explanation.json',
    'logs/target_resolution_all.json',
    'logs/target_intent.json',
    'logs/geocoder_referee.json',
    'logs/candidate_semantic_review.json',
    'logs/candidate_coordinate_enrichment.json',
    'logs/structured_endpoint_discovery.jsonl',
    'logs/promoted_asset_host_rows.jsonl',
    'logs/playwright_network_capture_errors.jsonl',
    'logs/output_summary.json',
]:
    p = OUTPUT_DIR / rel
    print(rel, 'exists=', p.exists(), 'size=', p.stat().st_size if p.exists() else 0)

explanation_md = OUTPUT_DIR / 'RUN_EXPLANATION.md'
if explanation_md.exists():
    display(Markdown(explanation_md.read_text(encoding='utf-8')))
else:
    print('No RUN_EXPLANATION.md was written. Inspect logs/run_summary.json for raw state.')

# Per-target diagnostics are written below logs/targets/<target_id>/ and candidates/<target_id>/.
for folder in sorted((OUTPUT_DIR / 'logs' / 'targets').glob('*')) if (OUTPUT_DIR / 'logs' / 'targets').exists() else []:
    print('target diagnostics:', folder.relative_to(OUTPUT_DIR))


## Camera candidate table

This cell loads `camera_candidates_table.csv` first. That CSV is written from all non-rejected review candidates, including candidates that do **not** have latitude/longitude and therefore cannot be mapped. If the CSV is missing, the cell falls back to `camera.geojson` or `untrusted_camera_candidates.geojson`.

In [ ]:
from pathlib import Path
from IPython.display import display

TABLE_PATH = OUTPUT_DIR / "camera_candidates_table.csv"
CAMERA_ROWS = []
GEOJSON_PATH = None

if TABLE_PATH.exists() and TABLE_PATH.stat().st_size > 0:
    print("Selected table:", TABLE_PATH)
    try:
        import pandas as pd
        df = pd.read_csv(TABLE_PATH)
        print("Rows:", len(df))
        display_cols = [
            "name", "target_label", "location_text", "camera_type", "camera_id", "latitude", "longitude",
            "stream_url", "source_url", "thumbnail_url", "media_type", "trust_level",
            "validation_status", "scope_status", "discovery_method", "coordinate_source",
            "review_required",
        ]
        existing_cols = [col for col in display_cols if col in df.columns]
        display(df[existing_cols].head(200))
        CAMERA_ROWS = df.to_dict("records")
        if {"latitude", "longitude"}.issubset(df.columns):
            coordinate_rows = df[df["latitude"].notna() & df["longitude"].notna()]
            print("Coordinate-bearing table rows:", len(coordinate_rows))
        else:
            print("Coordinate-bearing table rows: 0 (latitude/longitude columns missing)")
    except Exception as exc:
        print("Could not display candidate CSV table:", repr(exc))
else:
    print("No camera_candidates_table.csv found; falling back to GeoJSON.")
    from camera_discovery.utils.geojson_viewer import (
        load_camera_rows,
        select_camera_geojson,
        write_camera_table_csv,
    )
    GEOJSON_PATH = select_camera_geojson(OUTPUT_DIR)
    print("Selected GeoJSON:", GEOJSON_PATH)
    if GEOJSON_PATH is None:
        CAMERA_ROWS = []
        print("No trusted or untrusted camera GeoJSON found yet.")
    else:
        CAMERA_ROWS = load_camera_rows(GEOJSON_PATH)
        table_csv = write_camera_table_csv(OUTPUT_DIR, CAMERA_ROWS)
        print("Rows:", len(CAMERA_ROWS))
        print("CSV table:", table_csv)
        if not CAMERA_ROWS:
            print("GeoJSON exists but contains no camera features.")
        else:
            try:
                import pandas as pd
                df = pd.DataFrame(CAMERA_ROWS)
                display(df.head(200))
            except Exception:
                for row in CAMERA_ROWS[:25]:
                    print(row)


## Interactive camera map

The map below uses the selected trusted or untrusted GeoJSON. Click a marker to see camera metadata, including camera type and camera ID when available. HLS candidates get a video player button. Image snapshot candidates get a refreshing image viewer instead of a broken video player.

In Colab, the notebook uses an `IFrame` plus a direct file link fallback because inline HTML rendering can be blocked by the notebook environment.


In [ ]:
from IPython.display import IFrame, HTML, display
from camera_discovery.utils.geojson_viewer import select_camera_geojson, write_embedded_camera_map

MAP_GEOJSON_PATH = select_camera_geojson(OUTPUT_DIR)
print("Selected GeoJSON for map:", MAP_GEOJSON_PATH)

if MAP_GEOJSON_PATH is None:
    print("No GeoJSON available for map display yet. The table above may still contain non-coordinate candidates.")
else:
    MAP_PATH = write_embedded_camera_map(OUTPUT_DIR, MAP_GEOJSON_PATH, output_name="notebook_camera_map.html")
    print("Notebook map:", MAP_PATH)
    print("Open manually if the iframe is blank:", MAP_PATH.resolve())
    display(IFrame(src=str(MAP_PATH), width="100%", height=720))
    display(HTML(f'<p><a href="{MAP_PATH}" target="_blank">Open camera map in a new tab</a></p>'))
